# 🚗 Car Segmentation — Отчёт о проделанной работе

## Что было сделано

- Датасет: **Carvana Image Masking Challenge** (5088 изображений автомобилей)
- Архитектура: **UNet + ResNet34** энкодер предобученный на ImageNet
- Лосс: **WeightedBCE** (граница ×3) + **SoftDice**
- Ключевая идея: **Consistency Loss** — при обучении модель видела одно изображение сразу в трёх цветовых пространствах (RGB / LAB / HSV) и училась давать похожие предсказания для всех трёх
- При инференсе: **ансамбль 5 цветовых пространств** — RGB, LAB, HSV, YCrCb, HLS
- TTA: горизонтальный флип при каждом предсказании


## 0. Настройка

In [ ]:
import sys, os, cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torchvision.transforms.v2 as tfs_v2
from PIL import Image
from tqdm.notebook import tqdm as tqdm_nb

PROJECT_ROOT = r"D:\ML_2\Carvana\code\ver3\car-segmentation"
WEIGHTS_PATH = rf"{PROJECT_ROOT}\checkpoints\rgb\best_model.pth"

# ← Папки с тестовыми картинками
MY_TEST_DIR     = r"D:\ML_2\Carvana\my_test"    # твои картинки из интернета
KAGGLE_TEST_DIR = r"D:\ML_2\Carvana\test"        # картинки с Kaggle

IMG_SIZE = 512
sys.path.insert(0, PROJECT_ROOT)

from src.model import UNetModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
#  Все вспомогательные функции
# ============================================================

COLORSPACES = {
    "rgb":   {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
    "lab":   {"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
    "hsv":   {"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
    "ycrcb": {"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
    "hls":   {"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
}
CV2_CONV = {
    "rgb": None, "lab": cv2.COLOR_RGB2LAB, "hsv": cv2.COLOR_RGB2HSV,
    "ycrcb": cv2.COLOR_RGB2YCrCb, "hls": cv2.COLOR_RGB2HLS,
}
CS_LABELS = {"rgb": "RGB", "lab": "LAB", "hsv": "HSV", "ycrcb": "YCrCb", "hls": "HLS"}

def convert_image(img_rgb, cs):
    if cs == "rgb": return img_rgb
    return Image.fromarray(cv2.cvtColor(np.array(img_rgb), CV2_CONV[cs]))

def get_transform(cs):
    n = COLORSPACES[cs]
    return tfs_v2.Compose([
        tfs_v2.Resize((IMG_SIZE, IMG_SIZE)),
        tfs_v2.ToImage(),
        tfs_v2.ToDtype(torch.float32, scale=True),
        tfs_v2.Normalize(mean=n["mean"], std=n["std"]),
    ])

def predict_single(model, tensor, use_tta=True):
    with torch.no_grad():
        pred = torch.sigmoid(model(tensor.to(device)))
        if use_tta:
            pf = torch.sigmoid(model(torch.flip(tensor, dims=[3]).to(device)))
            pred = (pred + torch.flip(pf, dims=[3])) / 2
    return pred.cpu()

def remove_noise(mask_np, ratio):
    if ratio <= 0: return mask_np
    total = mask_np.shape[0] * mask_np.shape[1]
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_np, connectivity=8)
    clean = np.zeros_like(mask_np)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= int(total * ratio):
            clean[labels == i] = 255
    return clean

def to_mask(prob, thr=0.5, noise=0.01):
    t = prob
    while t.dim() > 2: t = t.squeeze(0)
    m = (t.numpy() * 255).astype(np.uint8)
    _, b = cv2.threshold(m, int(thr * 255), 255, cv2.THRESH_BINARY)
    return remove_noise(b, noise)

def get_masks(img_path, use_tta=True):
    img_rgb = Image.open(img_path).convert("RGB")
    display = np.array(img_rgb.resize((IMG_SIZE, IMG_SIZE)))
    preds, convs = {}, {}
    for cs in COLORSPACES:
        img_cs   = convert_image(img_rgb, cs)
        tensor   = get_transform(cs)(img_cs).unsqueeze(0)
        preds[cs] = predict_single(model, tensor, use_tta)
        convs[cs] = np.array(img_cs.resize((IMG_SIZE, IMG_SIZE)))
    return {"img": display, "convs": convs, "preds": preds}

def ensemble(d, css=None, wts=None, thr=0.5, noise=0.01):
    css = css or list(COLORSPACES)
    wts = wts or [1/len(css)] * len(css)
    tw  = sum(wts)
    avg = sum(d["preds"][cs] * (w/tw) for cs, w in zip(css, wts))
    mask = to_mask(avg.squeeze(0), thr, noise)
    car  = d["img"].copy()
    car[mask < 127] = 0
    return {"prob": avg, "mask": mask, "car": car}

def load_model(path):
    m = UNetModel(pretrained=False)
    m.load_state_dict(torch.load(path, map_location=device))
    return m.eval().to(device)

# Финальные параметры (все 5 пространств, равные веса)
FINAL_CSS = list(COLORSPACES)
FINAL_WTS = None  # равные
FINAL_THR = 0.5
FINAL_NOISE = 0.01

model = load_model(WEIGHTS_PATH)
print("Всё готово!")

---
# Часть 1. Как работает ансамбль цветовых пространств

Ключевое наблюдение: одна и та же модель RGB получает на вход изображение в разных цветовых форматах — и каждый раз видит машину немного по-другому. Усреднение всех пяти даёт самый чистый результат.

In [ ]:
# ← Возьми любую картинку из train для демонстрации
DEMO_IMG = r"D:\ML_2\Carvana\train\00087a6bd4dc_01.jpg"

d = get_masks(DEMO_IMG, use_tta=True)
print("Готово!")

In [ ]:
# Рис 1. Одно изображение — 5 разных представлений
cs_list = list(COLORSPACES)
fig, axes = plt.subplots(2, len(cs_list) + 1, figsize=(4 * (len(cs_list) + 1), 8))

# Оригинал
axes[0, 0].imshow(d["img"])
axes[0, 0].set_title("Оригинал (RGB)", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")
res_all = ensemble(d, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
axes[1, 0].imshow(res_all["mask"], cmap="gray")
axes[1, 0].set_title("Ансамбль всех 5", fontsize=11, fontweight="bold", color="darkgreen")
axes[1, 0].axis("off")

# Каждое пространство
for i, cs in enumerate(cs_list):
    axes[0, i+1].imshow(d["convs"][cs])
    axes[0, i+1].set_title(CS_LABELS[cs], fontsize=11)
    axes[0, i+1].axis("off")
    mask = to_mask(d["preds"][cs].squeeze(0), FINAL_THR, FINAL_NOISE)
    axes[1, i+1].imshow(mask, cmap="gray")
    axes[1, i+1].set_title(f"Маска {CS_LABELS[cs]}", fontsize=11)
    axes[1, i+1].axis("off")

plt.suptitle("Рис. 1 — Одна модель, 5 цветовых входов: каждый видит машину немного иначе",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("report_fig1_colorspaces.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Рис 2. Почему усреднение лучше — сравнение одиночных vs ансамбль
compare = [
    (["rgb"],                 [1.0], "Только RGB"),
    (["lab"],                 [1.0], "Только LAB"),
    (["hsv"],                 [1.0], "Только HSV"),
    (["rgb", "lab", "hsv"],   [0.5, 0.3, 0.2], "RGB + LAB + HSV"),
    (FINAL_CSS,               FINAL_WTS, "Все 5 (равные) ✓"),
]

n = len(compare)
fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))

for i, (css, wts, label) in enumerate(compare):
    res = ensemble(d, css, wts, FINAL_THR, FINAL_NOISE)
    color = "darkgreen" if "✓" in label else "black"
    axes[0, i].imshow(res["mask"], cmap="gray")
    axes[0, i].set_title(label, fontsize=10, color=color, fontweight="bold" if "✓" in label else "normal")
    axes[0, i].axis("off")
    axes[1, i].imshow(res["car"])
    axes[1, i].axis("off")

plt.suptitle("Рис. 2 — Усреднение цветовых пространств улучшает качество маски",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("report_fig2_ensemble.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Часть 2. Мой тестовый датасет

Картинки из интернета — разное качество, сложные условия съёмки, тени, нестандартные ракурсы.

In [ ]:
# Загружаем все картинки из my_test
my_imgs = sorted([
    os.path.join(MY_TEST_DIR, f)
    for f in os.listdir(MY_TEST_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])
print(f"Найдено картинок: {len(my_imgs)}")
for p in my_imgs:
    print(" ", os.path.basename(p))

In [ ]:
# Рис 3. Полный пайплайн на каждой картинке
my_results = {}
for path in tqdm_nb(my_imgs, desc="Обработка"):
    my_results[path] = get_masks(path, use_tta=True)

n = len(my_imgs)
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, path in enumerate(my_imgs):
    d_i  = my_results[path]
    res  = ensemble(d_i, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
    name = os.path.splitext(os.path.basename(path))[0]

    axes[i, 0].imshow(d_i["img"])
    axes[i, 0].set_title(f"Оригинал: {name}", fontsize=10)
    axes[i, 0].axis("off")

    # Лучший и худший одиночный канал для сравнения
    axes[i, 1].imshow(to_mask(d_i["preds"]["rgb"].squeeze(0), FINAL_THR, FINAL_NOISE), cmap="gray")
    axes[i, 1].set_title("Маска (только RGB)", fontsize=10)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res["mask"], cmap="gray")
    axes[i, 2].set_title("Маска (ансамбль 5)", fontsize=10, color="darkgreen", fontweight="bold")
    axes[i, 2].axis("off")

    # PNG с прозрачностью — показываем на сером фоне
    car_on_gray = np.full_like(d_i["img"], 180)
    alpha = res["mask"] / 255.0
    blended = (d_i["img"] * alpha[:, :, np.newaxis] +
               car_on_gray * (1 - alpha[:, :, np.newaxis])).astype(np.uint8)
    axes[i, 3].imshow(blended)
    axes[i, 3].set_title("Результат (серый фон)", fontsize=10)
    axes[i, 3].axis("off")

plt.suptitle("Рис. 3 — Мой тестовый датасет: картинки из интернета",
             fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig("report_fig3_my_test.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Рис 4. Детальный разбор одной сложной картинки
# ← Выбери самую интересную/сложную из своих
HARD_IMG = my_imgs[0]  # замени на индекс нужной картинки

d_hard = my_results[HARD_IMG]

fig, axes = plt.subplots(2, len(COLORSPACES) + 1, figsize=(4 * (len(COLORSPACES) + 1), 8))

axes[0, 0].imshow(d_hard["img"])
axes[0, 0].set_title("Оригинал", fontsize=11, fontweight="bold")
axes[0, 0].axis("off")
res_hard = ensemble(d_hard, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
axes[1, 0].imshow(res_hard["mask"], cmap="gray")
axes[1, 0].set_title("Ансамбль 5 ✓", fontsize=11, color="darkgreen", fontweight="bold")
axes[1, 0].axis("off")

for i, cs in enumerate(COLORSPACES):
    axes[0, i+1].imshow(d_hard["convs"][cs])
    axes[0, i+1].set_title(CS_LABELS[cs], fontsize=11)
    axes[0, i+1].axis("off")
    mask = to_mask(d_hard["preds"][cs].squeeze(0), FINAL_THR, FINAL_NOISE)
    axes[1, i+1].imshow(mask, cmap="gray")
    axes[1, i+1].set_title(f"Маска {CS_LABELS[cs]}", fontsize=11)
    axes[1, i+1].axis("off")

plt.suptitle(f"Рис. 4 — Детальный разбор сложной картинки: {os.path.basename(HARD_IMG)}",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("report_fig4_hard_case.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Часть 3. Тестовый датасет Kaggle

Картинки из соревнования — студийные фото с белым фоном, высокое качество.

In [ ]:
# Берём несколько случайных картинок из Kaggle test
import random
random.seed(42)

all_kaggle = sorted([
    os.path.join(KAGGLE_TEST_DIR, f)
    for f in os.listdir(KAGGLE_TEST_DIR)
    if f.lower().endswith((".jpg", ".jpeg"))
])

# ← Сколько картинок показать
N_KAGGLE = 6
kaggle_sample = random.sample(all_kaggle, min(N_KAGGLE, len(all_kaggle)))

print(f"Всего в Kaggle test: {len(all_kaggle)}")
print(f"Показываем:          {len(kaggle_sample)}")

In [ ]:
# Рис 5. Kaggle test — оригинал / маска / результат
kaggle_results = {}
for path in tqdm_nb(kaggle_sample, desc="Kaggle test"):
    kaggle_results[path] = get_masks(path, use_tta=True)

n = len(kaggle_sample)
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, path in enumerate(kaggle_sample):
    d_k  = kaggle_results[path]
    res  = ensemble(d_k, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
    name = os.path.basename(path)

    axes[i, 0].imshow(d_k["img"])
    axes[i, 0].set_title(f"{name}", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(res["mask"], cmap="gray")
    axes[i, 1].set_title("Маска (ансамбль 5)", fontsize=10)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(res["car"])
    axes[i, 2].set_title("Машина без фона", fontsize=10)
    axes[i, 2].axis("off")

plt.suptitle("Рис. 5 — Kaggle тестовый датасет: студийные фото с белым фоном",
             fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig("report_fig5_kaggle_test.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Рис 6. Сравнение: мои картинки vs Kaggle — влияние качества входа
# Берём по одной картинке из каждого набора
my_example     = my_imgs[0]
kaggle_example = kaggle_sample[0]

d_my     = my_results[my_example]
d_kaggle = kaggle_results[kaggle_example]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

titles_top = ["Оригинал", "LAB", "HSV", "Ансамбль 5 ✓"]
titles_bot = ["Оригинал", "LAB", "HSV", "Ансамбль 5 ✓"]

for row, (d_cur, label) in enumerate([(d_my, "Мой датасет (интернет)"),
                                       (d_kaggle, "Kaggle датасет (студия)")]):
    res = ensemble(d_cur, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
    show = [
        (d_cur["img"], "Оригинал"),
        (d_cur["convs"]["lab"], "LAB"),
        (d_cur["convs"]["hsv"], "HSV"),
        (res["car"], "Результат (ансамбль)"),
    ]
    for col, (img, title) in enumerate(show):
        axes[row, col].imshow(img if img.ndim == 3 else img, cmap="gray" if img.ndim == 2 else None)
        if col == 0:
            axes[row, col].set_ylabel(label, fontsize=12, fontweight="bold")
        axes[row, col].set_title(title, fontsize=10)
        axes[row, col].axis("off")

plt.suptitle("Рис. 6 — Мой датасет vs Kaggle: разные условия съёмки",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("report_fig6_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Итоги

| Компонент | Описание |
|---|---|
| Модель | UNet + ResNet34 (предобучен на ImageNet) |
| Датасет | Carvana, 5088 изображений |
| Лосс | WeightedBCE (граница ×3) + SoftDice |
| Consistency Loss | RGB + LAB + HSV при обучении |
| TTA | Горизонтальный флип |
| Ансамбль | 5 цветовых пространств с равными весами |
| IoU на валидации | ~0.89 |

**Главный вывод:** усреднение предсказаний модели по 5 цветовым пространствам (RGB, LAB, HSV, YCrCb, HLS) даёт более чистые маски чем любой одиночный вход, особенно на сложных случаях с тенями и нестандартным освещением.

In [ ]:
# Сохраняем все результаты на диск
OUT_DIR = r"D:\ML_2\Carvana\report_results"
os.makedirs(OUT_DIR, exist_ok=True)

# Мои картинки
for path in tqdm_nb(my_imgs, desc="Сохранение my_test"):
    name = os.path.splitext(os.path.basename(path))[0]
    d_i  = my_results[path]
    res  = ensemble(d_i, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
    cv2.imwrite(os.path.join(OUT_DIR, f"{name}_mask.png"), res["mask"])
    rgba = np.dstack([d_i["img"], res["mask"]])
    Image.fromarray(rgba.astype(np.uint8), mode="RGBA").save(
        os.path.join(OUT_DIR, f"{name}_no_bg.png"))

# Kaggle sample
for path in tqdm_nb(kaggle_sample, desc="Сохранение kaggle"):
    name = os.path.splitext(os.path.basename(path))[0]
    d_k  = kaggle_results[path]
    res  = ensemble(d_k, FINAL_CSS, FINAL_WTS, FINAL_THR, FINAL_NOISE)
    cv2.imwrite(os.path.join(OUT_DIR, f"kaggle_{name}_mask.png"), res["mask"])
    rgba = np.dstack([d_k["img"], res["mask"]])
    Image.fromarray(rgba.astype(np.uint8), mode="RGBA").save(
        os.path.join(OUT_DIR, f"kaggle_{name}_no_bg.png"))

print(f"\nВсё сохранено в: {OUT_DIR}")